# Creating artificial storm

In [1]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split         

In [2]:
import sys
sys.path.insert(1, "../../module")
import numpy as np      
import snflics

# Importing data needed to compute the average wavelet power and size

In [3]:
test = pd.read_csv("../../output/data/dakar/data-test-eps-dakar.csv")
train = pd.read_csv("../../output/data/dakar/data-train-eps-dakar.csv")
train, validation = train_test_split(train, test_size=0.2, random_state=12)

In [4]:
def choose_header(n, location, lead_time):
    """
    Generate header names for n closest storms.

    Returns:
        tuple of two lists:
            - input headers: year, month, day, hour, minute, lat1..n, lon1..n, wp1..n, size1..n, d1..n, mask1..n
            - target header: Cb_{location}_t{lead_time}
    """
    input_headers = ['year', 'month', 'day', 'hour', 'minute']
    
    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        input_headers.extend([f'{prefix}{i}' for i in range(1, n + 1)])
        
    return input_headers

In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
        """
        Compute Haversine distance between two points or arrays of points.
        Inputs are in degrees. Output is in kilometers.
        
        Supports both scalar and array inputs (NumPy).
        """
        R = 6371.0  # Earth radius in kilometers

        # Convert degrees to radians
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)

        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad

        a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

        return R * c

In [6]:
Dakar_lon = -17.467686
Dakar_lat = 14.716677

In [7]:
mean_wp = 3430.40
mean_size = 7460.30

In [8]:
# Get all wp and size values as flattened arrays
wp_values = train[[f"wp{i}" for i in range(1, 4)]].values.flatten()
size_values = train[[f"size{i}" for i in range(1, 4)]].values.flatten()

# Remove zeros before averaging
avg_wp = wp_values[wp_values > 0].mean()
avg_size = size_values[size_values > 0].mean()

In [16]:
np.median(size_values)

387.0

In [10]:
year = 2020
month = 9
day = 5
hour = 15
minute = 0

In [11]:
resolution = 0.01

In [12]:
longitude = np.arange(-20, -12, resolution)
latitude = np.arange(11, 19, resolution)

lons, lats = np.meshgrid(longitude, latitude)

In [ ]:
rng_offset = np.random.RandomState(42)

generated_data = []

for lat, lon in zip(lats.flatten(), lons.flatten()):
    storm0 = {'lat': lat, 'lon': lon}

    def random_offset(std_deg=0.2):
        return rng_offset.normal(0, std_deg)

    storm1 = {'lat': lat + random_offset(), 'lon': lon + random_offset()}
    storm2 = {'lat': lat + random_offset(), 'lon': lon + random_offset()}

    for storm in [storm0, storm1, storm2]:
        storm['wp'] = np.median(wp_values)
        storm['size'] = np.median(size_values) / 3
        storm['distance'] = haversine_distance(storm['lat'], storm['lon'], Dakar_lat, Dakar_lon)
        storm['mask'] = 1

    entry = {'year': year, 'month': month, 'day': day, 'hour': hour, 'minute': minute}

    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        for i, storm in enumerate([storm0, storm1, storm2], start=1):
            key = f"{prefix}{i}"
            entry[key] = storm['distance'] if prefix == 'd' else storm[prefix]

    generated_data.append(entry)

pd.DataFrame(generated_data).to_csv(f'./data/single/artificial-mcs-data-{resolution}.csv', index=False)
